# Casys ORM - Advanced Queries

This notebook demonstrates advanced query patterns:
- Complex WHERE conditions
- Aggregations (COUNT, SUM, AVG, etc.)
- GROUP BY
- Edge patterns and traversals
- Relationship navigation

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'python'))

import casys_db
from casys_db import NodeEntity, HasMany, HasOne, Session
import tempfile

## Setup

In [ ]:
class Person(NodeEntity):
    labels = ["Person"]
    name: str
    age: int
    city_name: str
    friends = HasMany("Person", via="KNOWS")

# Initialize
tmpdir = tempfile.mkdtemp()
engine = casys_db.CasysEngine(tmpdir)
engine.open_database("social")
try:
    engine.create_branch("social", "main")
except:
    pass
branch = engine.open_branch("social", "main")
session = Session(branch)

print("✓ Setup complete")

## 1. Create Sample Data

In [ ]:
# Create people
people_data = [
    {"name": "Alice", "age": 30, "city_name": "Paris"},
    {"name": "Bob", "age": 25, "city_name": "Paris"},
    {"name": "Charlie", "age": 35, "city_name": "London"},
    {"name": "Diana", "age": 28, "city_name": "Paris"},
    {"name": "Eve", "age": 32, "city_name": "London"},
]

people = []
for data in people_data:
    person = Person(**data)
    person_id = session.save(person)
    people.append(person)
    print(f"✓ Created {data['name']} (ID: {person_id})")

print(f"\n✓ Created {len(people)} people")

## 2. Complex WHERE Conditions

In [ ]:
# Multiple AND conditions
query = session.query(Person).where(
    lambda p: p.age > 25 and p.city_name == 'Paris'
)

print("Query: People over 25 in Paris")
print(f"GQL: {query._build_gql()}")
print(f"Results: {query.all()}")

In [ ]:
# Age range query
query = session.query(Person).where(
    lambda p: p.age >= 28 and p.age <= 32
)

print("Query: People aged 28-32")
print(f"GQL: {query._build_gql()}")
print(f"Results: {query.all()}")

## 3. Aggregations

In [ ]:
# COUNT
total = session.query(Person).count()
print(f"Total people: {total}")

# COUNT with filter
paris_count = session.query(Person).where(lambda p: p.city_name == 'Paris').count()
print(f"People in Paris: {paris_count}")

In [ ]:
# Raw aggregation queries
queries = [
    "MATCH (p:Person) RETURN COUNT(p) AS total",
    "MATCH (p:Person) RETURN AVG(p.age) AS avg_age",
    "MATCH (p:Person) RETURN MIN(p.age) AS min_age, MAX(p.age) AS max_age",
    "MATCH (p:Person) RETURN SUM(p.age) AS total_age",
]

for gql in queries:
    result = session.execute_raw(gql)
    print(f"\n{gql}")
    print(f"Result: {result}")

## 4. GROUP BY Aggregations

In [ ]:
# Count people by city
gql = "MATCH (p:Person) RETURN p.city_name, COUNT(p) AS count ORDER BY count DESC"
result = session.execute_raw(gql)

print("People per city:")
print(f"Columns: {result['columns']}")
for row in result['rows']:
    print(f"  {row}")

In [ ]:
# Average age by city
gql = "MATCH (p:Person) RETURN p.city_name, AVG(p.age) AS avg_age ORDER BY avg_age DESC"
result = session.execute_raw(gql)

print("Average age per city:")
print(f"Columns: {result['columns']}")
for row in result['rows']:
    print(f"  {row}")

## 5. Sorting and Limiting

In [ ]:
# Top 3 oldest people
query = session.query(Person).order_by_desc(lambda p: p.age).take(3)

print("Top 3 oldest people:")
print(f"GQL: {query._build_gql()}")
print(f"Results: {query.all()}")

In [ ]:
# Youngest person
youngest = session.query(Person).order_by(lambda p: p.age).first()

print(f"Youngest person: {youngest}")

## 6. Edge Patterns and Traversals

In [ ]:
# Create some friendships (edges)
# Note: This requires the add_edge API
if len(people) >= 3:
    # Alice knows Bob
    edge_id1 = branch.add_edge(people[0]._id, people[1]._id, "KNOWS", {"since": 2020})
    print(f"✓ Created edge: Alice -> Bob (ID: {edge_id1})")
    
    # Bob knows Charlie
    edge_id2 = branch.add_edge(people[1]._id, people[2]._id, "KNOWS", {"since": 2021})
    print(f"✓ Created edge: Bob -> Charlie (ID: {edge_id2})")
    
    # Alice knows Diana
    edge_id3 = branch.add_edge(people[0]._id, people[3]._id, "KNOWS", {"since": 2019})
    print(f"✓ Created edge: Alice -> Diana (ID: {edge_id3})")

In [ ]:
# Query: Find Alice's friends
gql = "MATCH (a:Person)-[:KNOWS]->(f:Person) WHERE a.name = 'Alice' RETURN f.name, f.age"
result = session.execute_raw(gql)

print("Alice's friends:")
print(f"Columns: {result['columns']}")
for row in result['rows']:
    print(f"  {row}")

In [1]:
# Query: Friends of friends (2 hops)
gql = "MATCH (a:Person)-[:KNOWS*2..2]->(fof:Person) WHERE a.name = 'Alice' RETURN fof.name, fof.age"
result = session.execute_raw(gql)

print("Alice's friends of friends (2 hops):")
print(f"Columns: {result['columns']}")
for row in result['rows']:
    print(f"  {row}")

NameError: name 'session' is not defined

In [ ]:
# Query: All reachable people (1 to 3 hops)
gql = "MATCH (a:Person)-[:KNOWS*1..3]->(p:Person) WHERE a.name = 'Alice' RETURN DISTINCT p.name, p.age"
result = session.execute_raw(gql)

print("People reachable from Alice (1-3 hops):")
print(f"Columns: {result['columns']}")
for row in result['rows']:
    print(f"  {row}")

## 7. Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("✓ Cleaned up")

## Summary

This notebook demonstrated:

1. **Complex Queries**: Multiple WHERE conditions with AND/OR
2. **Aggregations**: COUNT, SUM, AVG, MIN, MAX
3. **GROUP BY**: Aggregations grouped by properties
4. **Sorting**: ORDER BY with ASC/DESC
5. **Edge Patterns**: Creating and querying relationships
6. **Traversals**: Multi-hop queries with depth control (1..3, 2..2, etc.)

### Key Features

- **ISO GQL Compliant**: All queries follow ISO GQL standard
- **Depth Control**: `*1..3` syntax for multi-hop traversals
- **Flexible Aggregations**: Support for all standard aggregate functions
- **Edge Properties**: Relationships can have properties (e.g., `since: 2020`)